[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaronJenish/zawdwadw/blob/main/Experiment_16_LSTM_and_RNN_Performance_Comparison.ipynb)


# Experiment 16: LSTM and RNN Performance Comparison

CSA47 – Deep Learning. Run the setup cell first, then execute the experiment cells in order.


In [ ]:
# Common setup – run this cell first
import os, time, warnings, math
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, mean_absolute_error, mean_squared_error)
np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)
print("Setup complete")

## Experiment 16 – LSTM and RNN Performance Comparison

In [ ]:
import tensorflow as tf, time
t=np.arange(730); series=100+20*np.sin(2*np.pi*t/7)+.03*t+10*np.sin(2*np.pi*t/365)+np.random.normal(0,4,730)
sc=MinMaxScaler(); s=sc.fit_transform(series.reshape(-1,1)).ravel()
X,y=windows(s,30); cut=int(.8*len(X)); Xtr,Xte,ytr,yte=X[:cut],X[cut:],y[:cut],y[cut:]
rows=[]; preds={}
for name,layer in [("SimpleRNN",tf.keras.layers.SimpleRNN(32)),("LSTM",tf.keras.layers.LSTM(32))]:
    model=tf.keras.Sequential([tf.keras.layers.Input((30,1)),layer,tf.keras.layers.Dense(1)])
    model.compile("adam","mse"); start=time.time(); h=model.fit(Xtr,ytr,epochs=10,batch_size=32,validation_split=.1,verbose=0); elapsed=time.time()-start
    p=model.predict(Xte,verbose=0).ravel(); preds[name]=sc.inverse_transform(p.reshape(-1,1)).ravel()
    actual=sc.inverse_transform(yte.reshape(-1,1)).ravel()
    mae=mean_absolute_error(actual,preds[name]); rmse=np.sqrt(mean_squared_error(actual,preds[name]))
    mape=np.mean(np.abs((actual-preds[name])/np.maximum(np.abs(actual),1e-6)))*100
    rows.append([name,mae,rmse,mape,elapsed,h.history["loss"][-1],h.history["val_loss"][-1]])
print(pd.DataFrame(rows,columns=["Model","MAE","RMSE","MAPE %","Seconds","Train loss","Val loss"]))
plt.plot(actual,label="Actual")
for name,p in preds.items(): plt.plot(p,label=name)
plt.legend(); plt.title("Forecast comparison"); plt.show()